### Overview

When working with optical satellite imagery, we need to ensure the cloudy-pixels are removed from analysis. Most providers supply QA bands detailing locations of cloudy pixels. There are also third-party cloud-masking packages that can be used to locate and mask cloudy pixels.

In this notebook, we compare two cloud-masking approaches across a full time series of Sentinel-2 imagery: the Scene Classification (SCL) band supplied with Sentinel-2 Level-2A images, and [OmniCloudMask](https://github.com/DPIRD-DMA/OmniCloudMask), a deep-learning based cloud and cloud-shadow segmentation package. Both are applied to the same stack of scenes loaded from the STAC catalog, and compared via the NDVI time series they produce at a point of interest.

### Setup

Determine our runtime environment.

In [1]:
import os

if 'COLAB_RELEASE_TAG' in os.environ:
    environment = 'colab'
    if os.environ.get('VERTEX_PRODUCT') == 'COLAB_ENTERPRISE':
        environment = 'colab_enterprise'
else:
    environment = 'local'

print(f'Environment: {environment}')

Environment: local


If we are on Google Colab, install the required packages. Local runtimes are expected to have the packages already installed.

In [2]:
%%capture
if environment in ['colab', 'colab_enterprise']:
    !pip install pystac-client odc-stac rioxarray dask['distributed'] \
        jupyter-server-proxy odc-algo omnicloudmask

Import all required libraries. Make sure to import everything at the beginning as certain Xarray extensions are activated on import and registers certain accesors, like `.rio` and `.odc` for Xarray objects.

In [3]:
import dask
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import pystac_client
import rioxarray as rxr
import torch
import xarray as xr
from matplotlib.colors import ListedColormap
from odc.stac import configure_s3_access, load
from omnicloudmask import predict_from_array

Setup a local Dask cluster. This distributes the computation across multiple workers on your computer.

In [ ]:
from dask.distributed import Client

# As we are using GPU inference, we use a single worker client
client = Client(
    n_workers=1,
    threads_per_worker=2,
)
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 1
Total threads: 2,Total memory: 8.00 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:62244,Workers: 0
Dashboard: http://127.0.0.1:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:62249,Total threads: 2
Dashboard: http://127.0.0.1:62250/status,Memory: 8.00 GiB
Nanny: tcp://127.0.0.1:62247,


If you are running this notebook in Colab, you will need to create and use a proxy URL to see the dashboard running on the local server.

In [5]:
if environment == 'colab':
    from google.colab import output
    port_to_expose = 8787  # This is the default port for Dask dashboard
    print(output.eval_js(f'google.colab.kernel.proxyPort({port_to_expose})'))

### Masking a Single Scene

If a scene fits into memory, this approach is the most straightforward one. Let's learn this approach first and then we will see how we can scale this to a time-series.


We define a location and time of interest to get some satellite imagery.

In [9]:
latitude = 27.163
longitude = 82.608
year = 2023

Get a single cloudy scene.

In [16]:
# Define a GeoJSON geometry
geometry = {
    'type': 'Point',
    'coordinates': [longitude, latitude]
}

# Query the STAC Catalog
catalog = pystac_client.Client.open(
    'https://earth-search.aws.element84.com/v1')

search = catalog.search(
    collections=['sentinel-2-c1-l2a'],
    intersects=geometry,
    datetime=f'{year}',
    query={
        'eo:cloud_cover': {'lt': 50},
        's2:nodata_pixel_percentage': {'lt': 10}},
    sortby=[
        {'field': 'properties.eo:cloud_cover',
         'direction': 'desc'}
        ]
)
items = search.item_collection()

# Items were sorted in descending order of cloud cover,
# so the first item is the most cloudy
most_cloudy = items[0]

ds = load(
    [most_cloudy],
    bands=['red', 'green', 'blue', 'nir', 'scl'],
    resolution=10, # Load the data at lower resolution to speed up processing 
    crs='utm',
    chunks={'x': 1024, 'y': 1024},  # Explicitly define chunk sizes
    groupby='solar_day',
    preserve_original_order=True
)

scene = ds.squeeze()
# Mask nodata values
scene = scene.where(scene != 0)

# Apply scale/offset
scale = 0.0001
offset = -0.1
# Select spectral bands (all except 'scl')
data_bands = [band for band in scene.data_vars if band != 'scl']
for band in data_bands:
  scene[band] = scene[band] * scale + offset
scene

<xarray.Dataset> Size: 2GB
Dimensions:      (y: 10980, x: 10980)
Coordinates:
  * y            (y) float64 88kB 3.1e+06 3.1e+06 3.1e+06 ... 2.99e+06 2.99e+06
  * x            (x) float64 88kB 6e+05 6e+05 6e+05 ... 7.098e+05 7.098e+05
    spatial_ref  int32 4B 32644
    time         datetime64[us] 8B 2023-01-28T05:21:07.086000
Data variables:
    red          (y, x) float32 482MB dask.array<chunksize=(1024, 1024), meta=np.ndarray>
    green        (y, x) float32 482MB dask.array<chunksize=(1024, 1024), meta=np.ndarray>
    blue         (y, x) float32 482MB dask.array<chunksize=(1024, 1024), meta=np.ndarray>
    nir          (y, x) float32 482MB dask.array<chunksize=(1024, 1024), meta=np.ndarray>
    scl          (y, x) float32 482MB dask.array<chunksize=(1024, 1024), meta=np.ndarray>

Let’s call `compute()` to kick-off the dask graph. Dask will query the cloud-hosted dataset to fetch the required pixels. Once you run the cell, look at the Dask Diagnostic Dashboard to see the data processing in action.

In [17]:
%%time
scene = scene.compute()

CPU times: user 8.64 s, sys: 5.68 s, total: 14.3 s
Wall time: 2min 45s


### Apply SCL Mask

We use the bundled  Scene Classification (SCL) band for masking cloudy pixels. We select the types of pixels we want to mask. Let's create a mask that will remove all pixels marked `Cloud shadows (3)`, `Cloud Medium Probability (8)`, `Cloud High Probability (9)` and `Thin Cirrus (10)`.

In [18]:
mask = scene['scl'].isin([3,8,9,10])
# Apply the mask to all the data bands
scene_masked_scl = scene[data_bands].where(~mask)
scene_masked_scl

<xarray.Dataset> Size: 2GB
Dimensions:      (y: 10980, x: 10980)
Coordinates:
  * y            (y) float64 88kB 3.1e+06 3.1e+06 3.1e+06 ... 2.99e+06 2.99e+06
  * x            (x) float64 88kB 6e+05 6e+05 6e+05 ... 7.098e+05 7.098e+05
    spatial_ref  int32 4B 32644
    time         datetime64[us] 8B 2023-01-28T05:21:07.086000
Data variables:
    red          (y, x) float32 482MB 0.0211 0.0182 0.0239 ... nan nan nan
    green        (y, x) float32 482MB 0.0285 0.0287 0.0361 ... nan nan nan
    blue         (y, x) float32 482MB 0.0232 0.025 0.0282 0.0259 ... nan nan nan
    nir          (y, x) float32 482MB 0.1389 0.1435 0.1852 ... nan nan nan

### Apply OmniCloudMask

[OmniCloudMask](https://omnicloudmask.readthedocs.io/en/latest/index.html) uses a pre-trained Deep Learning model to predict clouds from satellite imagery. It supports resolutions from 10 m to 50 m and works with any imagery that has Red, Green, and NIR bands.

Using this method requires using a Neural Network to predict the mask and hence is it best run on a machine with a GPU. We first check and configure the backend available and download the model.

In [19]:
%%time
inference_device = (
    'cuda' if torch.cuda.is_available()
    else 'mps' if torch.backends.mps.is_available()
    else 'cpu'
)
print(f'Inference device: {inference_device}')

# Run a test prediction 
# This will download the model weights locally on the first call
predict_from_array(
    np.zeros((3, 50, 50), dtype=np.float32), inference_device=inference_device
)

Inference device: mps
CPU times: user 139 ms, sys: 225 ms, total: 364 ms
Wall time: 1.34 s


/opt/miniconda3/envs/python_remote_sensing/lib/python3.14/site-packages/omnicloudmask/cloud_mask.py:274: UserWarning: Significant no-data areas detected. Adjusting patch size to 32px and overlap to 16px to minimize no-data patches.
  patch_overlap, patch_size = check_patch_size(
/opt/miniconda3/envs/python_remote_sensing/lib/python3.14/site-packages/omnicloudmask/cloud_mask.py:274: UserWarning: Patch size 32px is less than 50px. Small patch sizes may not provide adequate spatial context for the model.
  patch_overlap, patch_size = check_patch_size(


array([[[0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        ...,
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0]]], shape=(1, 50, 50), dtype=uint8)

Select the required bands from our scene and use OmniCloudMask's [predict_from_array()](https://omnicloudmask.readthedocs.io/en/latest/api.html#predict-from-array) function to get the predicted cloud mask.

In [ ]:
scene_da = scene.to_array('band')
ocm_array = scene_da.sel(band=['red', 'green', 'nir'])

In [ ]:
%%time
scene_array = scene[['red', 'green', 'nir']].to_array('band')
ocm_mask = predict_from_array(ocm_array,  inference_device=inference_device)


### Visualize the Scene

Let's visualize and compare both the masks.


In [ ]:
scl_mask_da = scl_mask.astype('uint8')
ocm_mask_da = ocm_mask.astype('uint8')

scene_preview = scene_da.rio.reproject(
    scene_da.rio.crs, resolution=100
)

scl_mask_preview = scl_mask_da.rio.reproject(
    scl_mask_da.rio.crs, resolution=100
)

ocm_mask_preview = ocm_mask_da.rio.reproject(
    ocm_mask_da.rio.crs, resolution=100
)

In [ ]:

fig, (ax0, ax1, ax2) = plt.subplots(1, 3)
fig.set_size_inches(15,5)
scene_preview.sel(band=['red', 'green', 'blue']).plot.imshow(
    ax=ax0,
    vmin=0, vmax=0.3)
ax0.set_title('RGB Visualization')

# RGBA: Transparent, Red
mask_colormap = ListedColormap(['#00000000', '#FF0000FF'])
scl_mask_preview.plot.imshow(
    ax=ax1,
    cmap=mask_colormap,
    add_colorbar=False)

ax1.set_title('SCL Cloud Mask')
ocm_mask_preview.plot.imshow(
    ax=ax2,
    cmap=mask_colormap,
    add_colorbar=False)
ax2.set_title('OmniCloudMask Cloud Mask')

for ax in (ax0, ax1, ax2):
  ax.set_axis_off()
  ax.set_aspect('equal')
plt.show()

#### OmniCloudMask across the time series

Unlike SCL, OmniCloudMask cannot be broadcast over `time`, it runs a model on one full scene at a time. A plain Python loop that calls `.compute()` for each date would work, but it breaks out of the lazy Dask pipeline and holds every mask in memory at once, which is exactly what we want to avoid when everything else here is lazy and Dask-backed.

Instead we use [`dask.array.map_blocks`](https://docs.dask.org/en/stable/generated/dask.array.core.map_blocks.html) on the underlying Dask array. This turns the per-scene prediction into a node in the Dask graph: the whole thing stays lazy, results are streamed rather than accumulated, and it composes directly with the composite step below. Because the model needs a whole scene at once, we make `y`, `x` and `band` single chunks and set `time` to one scene per chunk so each date is an independent task.

**How expensive is one scene?** OmniCloudMask runs a model per scene, so it is worth knowing the per-scene cost before scaling to a full stack. We pull the first date from the time-series cube and time three back-to-back predictions on it (the weights are already downloaded, so this measures the steady-state per-call cost).

If you are curious, `predict_from_array` re-instantiates its model on every call, so we also checked whether preloading the model once and reusing it via its `custom_models` argument would speed things up. On this runtime it made no measurable difference &mdash; the per-scene time is dominated by inference itself, not model setup &mdash; so we keep the simple call below rather than depend on an internal helper for no gain.

In [ ]:
%%time
inference_device = 'cuda' if torch.cuda.is_available() else 'cpu'
#inference_device = 'mps'
# Download/cache the model weights once here, in the main process.
# Without this, the first call to predict_from_array happens inside
# map_blocks, where multiple worker processes race to download the
# same weights file concurrently, causing a FileNotFoundError.
predict_from_array(
    np.zeros((3, 32, 32), dtype=np.float32), inference_device=inference_device
)

# OmniCloudMask needs the full spatial extent of a scene at once (it
# tiles internally), so y/x/band must be single chunks; we chunk
# `time` to 1 so each date becomes an independent task in the graph.
scene_da = scene_da.chunk(band=-1)

ocm_input = ds[['red', 'green', 'nir']] \
    .to_array('band') \
    .chunk({'time': 1, 'band': -1})


def _ocm_predict_block(arr):
  # arr arrives as (band, time=1, y, x) — to_array() prepends 'band'.
  scene_arr = np.nan_to_num(arr[:, 0], nan=0.0).astype(np.float32)
  pred = predict_from_array(scene_arr,  inference_device=inference_device)
  return (pred[0] > 0)[np.newaxis]  # (1, y, x) boolean: True where cloud/shadow

# map_blocks runs the per-scene prediction as a Dask graph node,
# so the whole pipeline stays lazy instead of computing in a loop.
ocm_mask_data = ocm_input.data.map_blocks(
    _ocm_predict_block,
    drop_axis=0,  # band axis is consumed by the prediction
    dtype=bool,
)
ocm_mask_ts = xr.DataArray(
    ocm_mask_data,
    dims=('time', 'y', 'x'),
    coords={'time': ocm_input.time, 'y': ocm_input.y, 'x': ocm_input.x},
).rio.write_crs(scene_da.rio.crs)  # Copy CRS from the input data

In [ ]:
scene_mask = ocm_mask_ts.sel(time=most_cloudy.datetime.replace(tzinfo=None))
scene_mask

In [ ]:
%%time
scene_mask = scene_mask.compute()

In [ ]:
scene_mask = scene_mask.astype('uint8')  # Convert boolean to uint8 for visualization

In [ ]:
scene_preview = scene_da.rio.reproject(
    scene_da.rio.crs, resolution=100
)
mask_preview = scene_mask.rio.write_crs(scene_da.rio.crs).rio.reproject(
    scene_mask.rio.crs, resolution=100
)
fig, (ax0, ax1) = plt.subplots(1, 2)
fig.set_size_inches(10,5)
scene_preview.sel(band=['red', 'green', 'blue']).plot.imshow(
    ax=ax0,
    vmin=0, vmax=3000)
ax0.set_title('RGB Visualization')

# RGBA: Transparent, Red
mask_colormap = ListedColormap(['#00000000', '#FF0000FF'])
mask_preview.plot.imshow(
    ax=ax1,
    cmap=mask_colormap,
    add_colorbar=False)

ax1.set_title('Cloud Mask')
for ax in (ax0, ax1):
  ax.set_axis_off()
  ax.set_aspect('equal')
plt.show()

In [ ]:
ts_masked_ocm = ds.where(~ocm_mask_ts)
ts_masked_ocm

In [ ]:
ts_mask = ds['scl'].isin([3, 8, 9, 10])
ts_masked_scl = ds[data_bands].where(~ts_mask)
ts_masked_scl

### Compare the Two Methods as a Time Series

A composite collapses the whole stack into one image, which hides *when* each method masked a pixel. The difference between the two cloud masks shows up most clearly in an analysis-ready **time series**: for a single location we can see which dates each mask drops and how noisy the retained signal is.

Following the course's [time-series workflow](https://courses.spatialthoughts.com/python-remote-sensing.html#extracting-and-processing-time-series), we extract an NDVI time series at a point of interest from each masked stack and compare them.

In [ ]:
# NDVI from each cloud-masked stack. Masked (cloudy) pixels are NaN,
# so they propagate as NaN into the NDVI time series.
def compute_ndvi(ds):
    return (ds['nir'] - ds['red']) / (ds['nir'] + ds['red'])

ndvi_scl = compute_ndvi(ts_masked_scl)
ndvi_ocm = compute_ndvi(ts_masked_ocm)

We reproject our point of interest into the cube's CRS and sample the nearest pixel across all dates, exactly as the course does with `interp(..., method='nearest')`. Dates whose pixel was flagged as cloud or shadow are `NaN`, so they appear as gaps in the series.

**Note:** the OmniCloudMask series forces per-scene model inference for every date. At the course's 10&nbsp;m resolution over a full year this is heavy, so expect this cell to run for a while (if it is too slow or runs out of memory, lower the load `resolution` in the datacube cell above).

In [ ]:
from pyproj import Transformer

# Reproject the point of interest (defined near the top of the
# notebook) from lat/lon into the cube's UTM CRS.
transformer = Transformer.from_crs(
    'EPSG:4326', ds.rio.crs, always_xy=True)
x0, y0 = transformer.transform(longitude, latitude)

# Sample the nearest pixel across every date, as the course does.
ndvi_scl_pt = ndvi_scl.interp(x=x0, y=y0, method='nearest').compute()
ndvi_ocm_pt = ndvi_ocm.interp(x=x0, y=y0, method='nearest').compute()

Plot the two raw series together. Wherever a line breaks, that date was masked out by that method &mdash; so the gaps themselves show where SCL and OmniCloudMask disagree about clouds at this location.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(ndvi_scl_pt.time.values, ndvi_scl_pt.values,
        '-o', ms=5, lw=2, color='#1e88e5', alpha=0.75,
        label='SCL masked', zorder=2)
ax.plot(ndvi_ocm_pt.time.values, ndvi_ocm_pt.values,
        '--^', ms=5, lw=1.5, color='#ff9900', alpha=0.75,
        label='OmniCloudMask masked', zorder=3)

ax.set_title('NDVI time series at the point of interest')
ax.set_xlabel('Date'); ax.set_ylabel('NDVI')
ax.set_ylim(-0.2, 1)
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

The raw series is gappy because cloudy dates are removed. Following the course's processing, we regularize it to a fixed 5-day cadence, linearly interpolate the interior gaps, back-/forward-fill the leading and trailing gaps, and apply a short centered rolling mean. Doing this for both masks shows how the choice of cloud mask propagates into the final smoothed curve.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

# Raw observations as faint markers (kept as-is, just lighter so the
# smoothed lines read clearly on top)
ax.plot(ndvi_scl_pt.time.values, ndvi_scl_pt.values,
        'o', ms=4, color='#1e88e5', alpha=0.25, zorder=1)
ax.plot(ndvi_ocm_pt.time.values, ndvi_ocm_pt.values,
        '^', ms=4, color='#ff9900', alpha=0.25, zorder=1)

# Smoothed series, styled so overlap is still visible as two lines
# ax.plot(ndvi_scl_smooth.time.values, ndvi_scl_smooth.values,
#         '-o', ms=3, lw=2, color='#1e88e5', alpha=0.75,
#         label='SCL (smoothed)', zorder=3)
# ax.plot(ndvi_ocm_smooth.time.values, ndvi_ocm_smooth.values,
#         '--^', ms=3, lw=1.5, color='#ff9900', alpha=0.75,
#         label='OmniCloudMask (smoothed)', zorder=2)

ax.set_title('Smoothed NDVI time series — SCL vs OmniCloudMask')
ax.set_xlabel('Date')
ax.set_ylabel('NDVI')
ax.set_ylim(-0.2, 1)
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

**A note on scaling up and on `map_blocks`:**

* We measured the per-scene cost and confirmed it is dominated by inference. `predict_from_array` does re-instantiate the model on each call, but with weights cached that overhead is negligible here, so reusing a preloaded model via `custom_models` gave no measurable speedup. The per-scene inference time is the real, unavoidable cost.
* **On a single GPU, be careful with parallelism.** `map_blocks` may run several time-chunks at once across workers, all competing for the same GPU and risking out-of-memory errors. For GPU inference, use a single worker (or a one-worker `Client`); reserve cross-worker parallelism for CPU.
* For a fully supported, file-based batch workflow, OmniCloudMask's own `predict_from_load_func` loads the weights once and batches inference across scenes.

Close the dask client. This presents multiple clients being instantiated when running different notebooks on the same machine. This is not required on Colab but a good practice when you are running it on a local machine. Uncomment and run to shutdown the dask cluster.

In [ ]:
#client.shutdown()